# 長時間の学習を途中から再開できるようにする

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/A_colab_python_tips/colab_checkpoint.ipynb)

NLP クックブック A-09 ／ Colab と Python の小技

Colab は一定時間で切断される。学習の途中経過を Google Drive に保存して、切れても続きから再開したい。

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
CKPT_DIR = "/content/drive/MyDrive/Colab Notebooks/nlpcookbook/checkpoints/my-run"

In [ ]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir=CKPT_DIR,                 # チェックポイントを Drive に直接書く
    save_strategy="steps", save_steps=500,
    save_total_limit=2,                  # 古いものを消して Drive の容量を節約（最新の 2 つだけ残す）
    eval_strategy="steps", eval_steps=500,
    load_best_model_at_end=True,
    num_train_epochs=3, per_device_train_batch_size=32, report_to="none",
)
# trainer = Trainer(model=model, args=args, train_dataset=..., eval_dataset=...)

In [ ]:
import os
from transformers.trainer_utils import get_last_checkpoint

last = get_last_checkpoint(CKPT_DIR) if os.path.isdir(CKPT_DIR) else None
print("再開するチェックポイント:", last)
# trainer.train(resume_from_checkpoint=last)   # None なら最初から、あれば続きから（オプティマイザや学習率の状態も復元される）

In [ ]:
# 自前の学習ループの場合: モデルだけでなく、オプティマイザ・エポック数・乱数の状態も一緒に保存する
import torch

def save_checkpoint(path, model, optimizer, epoch, step):
    tmp = path + ".tmp"
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "epoch": epoch, "step": step,
                "torch_rng": torch.get_rng_state()}, tmp)
    os.replace(tmp, path)                # 書き込み途中で切断されても壊れたファイルが残らないようにする

def load_checkpoint(path, model, optimizer):
    ckpt = torch.load(path, map_location="cpu")
    model.load_state_dict(ckpt["model"]); optimizer.load_state_dict(ckpt["optimizer"]); torch.set_rng_state(ckpt["torch_rng"])
    return ckpt["epoch"], ckpt["step"]

- モデルの重みだけを保存して再開すると、オプティマイザの状態や学習率スケジュールが初期化されて、学習が乱れる。
- 保存の間隔は「失っても許せる時間」で決める（15〜30 分に 1 回が目安）。保存自体にも時間と Drive の容量がかかる。
- 無料枠は最長 12 時間、操作がないとさらに早く切れる。数時間を超える学習は、最初から再開を前提に組む。